# VSB temporal and CWT experts. This notebook launches the bounded Parquet-streaming VSB expert stage. The native 800,000-sample temporal input is retained, while the amended v2 protocol fixes physical and inference batch 4 for both experts; io-batch-size only limits Parquet/cache I/O.

In [ ]:
import os; import subprocess; import sys; from pathlib import Path; import torch; ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir() and (path / 'src').is_dir()); runtime_env = Path(os.environ.get('CONDA_PREFIX', '')).name; interpreter_env = Path(sys.executable).resolve().parent.parent.name; assert 'partial-discharge' in (runtime_env, interpreter_env), 'Use the partial-discharge mamba environment/kernel.'; assert torch.cuda.is_available(), 'CUDA is unavailable; training is blocked before launch.'; raw_root = Path(os.environ.get('PD_RAW_DATA_ROOT', str(ROOT / 'data/raw'))).resolve(); env = os.environ.copy(); env['PD_RAW_DATA_ROOT'] = str(raw_root); env['PYTHONPATH'] = str(ROOT / 'src'); config = ROOT / 'configs/experiments/two-dataset-confirmatory-v2-batch4-localraw.yaml'; io_batch_size = os.environ.get('PD_VSB_IO_BATCH_SIZE', '4'); command = [sys.executable, 'scripts/run_vsb_experts.py', '--config', str(config), '--io-batch-size', io_batch_size, '--seeds', '42', '43', '44', '45', '46']; print('Working directory:', ROOT); print('Environment:', runtime_env or interpreter_env); print('GPU:', torch.cuda.get_device_name(0)); print('Command:', ' '.join(command)); print('Neural batch size is 4; inference batch size is 4; Parquet I/O batch size:', io_batch_size); print('Set PD_RUN_EXPERIMENT=1 before executing this cell to launch training.'); subprocess.run(command, cwd=ROOT, env=env, check=True) if os.environ.get('PD_RUN_EXPERIMENT') == '1' else print('Dry wrapper only; no model, Parquet stream, or CUDA work was started.')

The runner keeps every id_measurement together, computes imbalance weights inside train folds, writes memmap caches, and excludes the unlabeled official test set from scientific metrics. Check the GPU budget before setting PD_RUN_EXPERIMENT=1.